In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

/bin/bash: /home/rkannan/miniconda3/envs/richard_tf/lib/libtinfo.so.6: no version information available (required by /bin/bash)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 149.6 kB/s eta 0:00:000:00:01m eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 298.8 kB/s eta 0:00:001m327.6 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 324.7 kB/s eta 0:00:001m345.8 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.5/450.5 kB 323.5 kB/s eta 0:00:000:00:010:00:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 527.4 kB/s eta 0:00:001m523.9 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 237.0 kB/s eta 0:00:00m eta 0:00:010:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 334.8 kB/s eta 0:00:00 kB/s eta 0:00:01
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.65.0
    Uninstalling tqdm-4.65.0:
      Successfully uninstalled tqdm-4.65.0
  Attempting

### You will need to setup git, adapt your email and name in the following cell.

In [53]:
!git config --global user.email "richannan24@gmail.com"
!git config --global user.name "Astonish24"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/bin/bash: /home/rkannan/miniconda3/envs/richard_tf/lib/libtinfo.so.6: no version information available (required by /bin/bash)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/bin/bash: /home/rkannan/miniconda3/envs/richard_tf/lib/libtinfo.so.6: no version information available (required by /bin/bash)


### You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [55]:
from huggingface_hub import notebook_login

notebook_login()

### Dataset

In [56]:
import numpy as np
import pandas as pd

In [217]:
import pandas as pd

# Read the CSV file
data = pd.read_csv('sample _questions_for_pilot_test.csv')

# Display the first few rows
data_df = pd.DataFrame(data)
raw_datasets = data_df
raw_datasets.dropna(inplace=True)
raw_datasets.head()

,question,answers,context
0,What is the primary focus of the integrative p...,The primary focus is to integrate sustainable ...,This integrative process seeks to incorporate ...
1,Why is human health prioritized in the integra...,Human health is prioritized to ensure that bui...,This integrative process seeks to incorporate ...
2,At what stage should the cross-discipline appr...,The cross-discipline approach should begin dur...,This integrative process seeks to incorporate ...
3,What document is prepared to establish the fou...,The Owner’s Project Requirements (OPR) document.,This integrative process seeks to incorporate ...
4,What is the purpose of the preliminary LEED me...,What is the purpose of the preliminary LEED mT...,This integrative process seeks to incorporate ...


In [218]:
from difflib import SequenceMatcher

def find_answer_start_position(answer, context):
    """
    Finds the start position of the answer in the context and returns it in a specific format.
    
    Parameters:
    answer (str): The answer string to search for.
    context (str): The context string to search in.
    
    Returns:
    dict: A dictionary in the format {'text': [answer], 'answer_start': [start_index]}.
    """
    match = SequenceMatcher(None, context, answer).find_longest_match(0, len(context), 0, len(answer))
    if match.size > 0:
        start_index = match.a  # Start index in the context
        return {'text': [answer], 'answer_start': [start_index]}
    return {'text': [answer], 'answer_start': [-1]}  # -1 indicates the answer was not found

# # Example usage
# answer = "The primary focus is to integrate sustainable and cost-effective green design and construction strategies with a core emphasis on human health"
# context = """This integrative process seeks to incorporate sustainable and cost-effective green design and construction strategies with a core focus on human health. By embedding health as a primary evaluative criterion in building design, construction, and operational strategies, this approach promotes innovative practices that foster high-performance healing environments for patients, caregivers, and staff."""

# result = find_answer_start_position(answer, context)
# print(result)

In [220]:
answer_index = []
for i in range(0, raw_datasets.shape[0]):
    answer_index.append(find_answer_start_position(raw_datasets.iloc[i]['answers'],raw_datasets.iloc[i]['question']))
raw_datasets['answer_idx'] = answer_index
# Assuming df is your DataFrame
unique_numbers = [f"unique-{i}" for i in range(raw_datasets.shape[0])]
raw_datasets['id'] = unique_numbers
raw_datasets.to_json()

'{"question":{"0":"What is the primary focus of the integrative process in healthcare projects?","1":"Why is human health prioritized in the integrative process?","2":"At what stage should the cross-discipline approach begin in healthcare projects?","3":"What document is prepared to establish the foundation for a health-focused environment in healthcare projects?","4":"What is the purpose of the preliminary LEED meeting in healthcare projects?","5":"When is it ideal to conduct the preliminary LEED meeting?","6":"How many professionals should the integrated project team include at minimum?","7":"Who might be part of the integrated project team in a healthcare project?","8":"What is the purpose of having a comprehensive project team in healthcare projects?","9":"What collaborative session must be held to enhance planning, and who should attend?","10":"How long should the design charrette session last?","11":"At what project phase is the design charrette ideally conducted?","12":"What is 

In [221]:
from datasets import Dataset, DatasetDict
import pandas as pd

# Assume raw_datasets is your pandas DataFrame
# Split the dataset into train and validation sets (adjust proportions as needed)
train_df = raw_datasets.sample(frac=0.8, random_state=42)  # 80% for training
validation_df = raw_datasets.drop(train_df.index)  # Remaining 20% for validation

# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
validation_dataset = Dataset.from_pandas(validation_df)

# Create the DatasetDict
dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset
})


# Display dataset summary
print(dataset_dict)
raw_datasets = dataset_dict

DatasetDict({
    train: Dataset({
        features: ['question', 'answers', 'context', 'answer_idx', 'id', '__index_level_0__'],
        num_rows: 64
    })
    validation: Dataset({
        features: ['question', 'answers', 'context', 'answer_idx', 'id', '__index_level_0__'],
        num_rows: 16
    })
})


In [222]:
print(raw_datasets['train'][0:1]['answer_idx'])

[{'answer_start': [128], 'text': ['Avoiding such land ensures the protection of resources essential for food production, ecological health, and water management.']}]


In [223]:
# from sklearn.model_selection import train_test_split

# # Split the data into training and validation sets
# raw_datasets, raw_datasets_val = train_test_split(
#     raw_datasets,  # The DataFrame containing the processed data
#     test_size=0.2,  # Percentage of the data to be used for validation (e.g., 20%)
#     random_state=42,  # For reproducibility
#     shuffle=True  # Shuffle the data before splitting
# )

# # Split the validation set into validation and test sets
# raw_datasets_val, raw_datasets_test = train_test_split(
#     raw_datasets_val,  # The previously created validation DataFrame
#     test_size=0.1,  # Split 50% of the validation set into test data
#     random_state=42,  # For reproducibility
#     shuffle=True  # Shuffle the data before splitting
# )

# # Display the sizes of the splits
# print(f"Training set size: {len(raw_datasets)}")
# print(f"Validation set size: {len(raw_datasets_val)}")
# print(f"Test set size: {len(raw_datasets_test)}")

In [224]:
print("Context: ", raw_datasets["train"][0]["context"])
print("Question: ", raw_datasets["train"][0]["question"])
print("Answer: ", raw_datasets["train"][0]["answer_idx"])

Context:  The Sensitive Land Protection credit is another important component of the LT credits, awarding 1 to 2 points to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options: locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, or habitats for endangered species. This protection extends to water bodies and wetlands, requiring a minimum buffer of 100 feet for water bodies and 50 feet for wetlands. Projects aiming for this credit must be careful to avoid land identified as critical for agriculture, ecosystems, or water retention. Sensitive habitats are defined by regulations like the U.S. Endangered Species Act, NatureServe classifications, or equivalent local standards, ensuring protection for vulnerable ecosystems and species. For projects on previously developed land or those that meet sensitive land standards, minor 

In [225]:
raw_datasets["train"].filter(lambda x: len(x["answer_idx"]["text"]) != 1)

Filter:   0%|          | 0/64 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answers', 'context', 'answer_idx', 'id', '__index_level_0__'],
    num_rows: 0
})

In [226]:
print(raw_datasets["validation"][0]["answer_idx"])
print(raw_datasets["validation"][2]["answer_idx"])

{'answer_start': [19], 'text': ['Human health is prioritized to ensure that building design, construction, and operational strategies foster high-performance healing environments for patients, caregivers, and staff.']}
{'answer_start': [16], 'text': ['The primary purpose of the Integrative Process credit is to enhance high-performance and cost-effective project outcomes by analyzing and leveraging the interrelationships among various building systems early in the design process.']}


In [227]:
print(raw_datasets["validation"][2]["context"])
print(raw_datasets["validation"][2]["question"])

The Integrative Process credit for BD&C projects aims to enhance high-performance, cost-effective outcomes by analyzing and leveraging the interrelationships among different building systems. This credit applies across various project types, including new construction, core and shell, schools, retail, data centers, warehouses, distribution centers, hospitality, and healthcare facilities. The process begins during the pre-design phase and continues through the design stages to maximize synergy across disciplines and building systems. By incorporating these interdisciplinary insights into essential project documents, such as the Owner’s Project Requirements (OPR) and Basis of Design (BOD), teams can ensure the resulting design aligns with both operational efficiency and sustainability goals.
What is the main purpose of the Integrative Process credit in BD&C projects?


In [228]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/home/rkannan/miniconda3/envs/richard_tf/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [229]:
tokenizer.is_fast

True

In [230]:
context = raw_datasets["train"][0]["context"]
question = raw_datasets["train"][0]["question"]

inputs = tokenizer(question, context)
tokenizer.decode(inputs["input_ids"])

'[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] The Sensitive Land Protection credit is another important component of the LT credits, awarding 1 to 2 points to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, or habitats for endangered species. This protection extends to water bodies and wetlands, requiring a minimum buffer of 100 feet for water bodies and 50 feet for wetlands. Projects aiming for this credit must be careful to avoid land identified as critical for agriculture, ecosystems, or water retention. Sensitive habitats are defined by regulations like the U. S. Endangered Species Act, NatureServe classifications, or equivalent local standards, e

In [231]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] The Sensitive Land Protection credit is another important component of the LT credits, awarding 1 to 2 points to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, [SEP]
[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, or habitat

In [232]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] The Sensitive Land Protection credit is another important component of the LT credits, awarding 1 to 2 points to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, [SEP]
[CLS] Why is it important to avoid land identified as critical for agriculture, ecosystems, or water retention in the Sensitive Land Protection credit? [SEP] to projects that avoid environmentally sensitive lands, thus minimizing the ecological impact of development. The credit offers two options : locating on previously developed land or choosing land that does not meet criteria for sensitivity, such as prime farmland, floodplains, or habitat

In [233]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)
inputs.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping'])

In [234]:
inputs["overflow_to_sample_mapping"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [235]:
inputs = tokenizer(
    raw_datasets["train"][2:6]["question"],
    raw_datasets["train"][2:6]["context"],
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

print(f"The 4 examples gave {len(inputs['input_ids'])} features.")
print(f"Here is where each comes from: {inputs['overflow_to_sample_mapping']}.")

The 4 examples gave 17 features.
Here is where each comes from: [0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3].


In [236]:
answers = raw_datasets["train"][2:6]["answer_idx"]
print(answers)

[{'answer_start': [3], 'text': ['By incorporating multifunctional spaces, projects can reduce the overall building area and optimize space usage, thereby lowering energy and water demands and creating a more efficient, adaptable design.']}, {'answer_start': [6], 'text': ['Minor improvements such as narrow bicycle paths, native plant restoration, and small clearings are allowed to support sustainable access and appreciation of these natural areas.']}, {'answer_start': [1], 'text': ['The water budget analysis is crucial for identifying methods to reduce potable water use and exploring nonpotable sources, optimizing systems like plumbing, rainwater management, and landscaping for greater water efficiency.']}, {'answer_start': [56], 'text': ['By awarding points to projects that avoid environmentally sensitive lands, the credit minimizes the ecological impact of development.']}]


In [237]:
answers = raw_datasets["train"][2:6]["answer_idx"]
start_positions = []
end_positions = []

for i, offset in enumerate(inputs["offset_mapping"]):
    sample_idx = inputs["overflow_to_sample_mapping"][i]
    answer = answers[sample_idx]
    start_char = answer["answer_start"][0]
    end_char = answer["answer_start"][0] + len(answer["text"][0])
    sequence_ids = inputs.sequence_ids(i)

    # Find the start and end of the context
    idx = 0
    while sequence_ids[idx] != 1:
        idx += 1
    context_start = idx
    while sequence_ids[idx] == 1:
        idx += 1
    context_end = idx - 1

    # If the answer is not fully inside the context, label is (0, 0)
    if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
        start_positions.append(0)
        end_positions.append(0)
    else:
        # Otherwise it's the start and end token positions
        idx = context_start
        while idx <= context_end and offset[idx][0] <= start_char:
            idx += 1
        start_positions.append(idx - 1)

        idx = context_end
        while idx >= context_start and offset[idx][1] >= end_char:
            idx -= 1
        end_positions.append(idx + 1)

start_positions, end_positions

([19, 0, 0, 0, 0, 19, 0, 0, 0, 18, 0, 0, 0, 23, 0, 0, 0],
 [60, 0, 0, 0, 0, 46, 0, 0, 0, 59, 0, 0, 0, 49, 0, 0, 0])

In [238]:
idx = 0
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

start = start_positions[idx]
end = end_positions[idx]
labeled_answer = tokenizer.decode(inputs["input_ids"][idx][start : end + 1])

print(f"Theoretical answer: {answer}, labels give: {labeled_answer}")

Theoretical answer: By incorporating multifunctional spaces, projects can reduce the overall building area and optimize space usage, thereby lowering energy and water demands and creating a more efficient, adaptable design., labels give: Energy - related systems are a core focus within the Integrative Process, requiring a preliminary energy modeling analysis, often called a “ simple box ” model, completed before the schematic design phase. This


In [239]:
idx = 4
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

decoded_example = tokenizer.decode(inputs["input_ids"][idx])
print(f"Theoretical answer: {answer}, decoded example: {decoded_example}")

Theoretical answer: By incorporating multifunctional spaces, projects can reduce the overall building area and optimize space usage, thereby lowering energy and water demands and creating a more efficient, adaptable design., decoded example: [CLS] In what ways might multifunctional spaces contribute to the Integrative Process? [SEP], thermal comfort ranges, and plug load demands are analyzed to identify opportunities for energy efficiency. Documentation of how these analyses inform decisions is then reflected in the project ’ s form, geometry, and building envelope features, often resulting in significant downsizing or simplification of building systems such as HVAC, lighting, and other controls. [SEP]


In [240]:
max_length = 384
stride = 128


def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answer_idx"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [241]:
train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
len(raw_datasets["train"]), len(train_dataset)

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

(64, 64)

In [242]:
print(train_dataset[0])

{'input_ids': [101, 2009, 1110, 1122, 1696, 1106, 3644, 1657, 3626, 1112, 3607, 1111, 6487, 117, 23600, 117, 1137, 1447, 23406, 1107, 1103, 14895, 22472, 4026, 8063, 4755, 136, 102, 1109, 14895, 22472, 4026, 8063, 4755, 1110, 1330, 1696, 6552, 1104, 1103, 149, 1942, 6459, 117, 25055, 122, 1106, 123, 1827, 1106, 3203, 1115, 3644, 4801, 1193, 7246, 4508, 117, 2456, 8715, 25596, 1103, 14769, 3772, 1104, 1718, 119, 1109, 4755, 3272, 1160, 6665, 131, 25338, 14829, 1113, 2331, 1872, 1657, 1137, 11027, 1657, 1115, 1674, 1136, 2283, 9173, 1111, 15750, 117, 1216, 1112, 5748, 17790, 117, 7870, 18220, 1116, 117, 1137, 10902, 1111, 11532, 1530, 119, 1188, 3636, 8559, 1106, 1447, 3470, 1105, 20432, 117, 8753, 170, 5867, 20232, 1104, 1620, 1623, 1111, 1447, 3470, 1105, 1851, 1623, 1111, 20432, 119, 21454, 14485, 1111, 1142, 4755, 1538, 1129, 5784, 1106, 3644, 1657, 3626, 1112, 3607, 1111, 6487, 117, 23600, 117, 1137, 1447, 23406, 119, 14895, 22472, 10902, 1132, 3393, 1118, 7225, 1176, 1103, 158, 119

In [244]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs
# example_ids = []
# def preprocess_validation_examples(examples, unique_id_start=1):
#     """
#     Preprocess validation examples and generate unique IDs for each example.

#     Parameters:
#     - examples: The dataset examples containing 'question', 'context', etc.
#     - unique_id_start: Starting value for generating unique IDs.

#     Returns:
#     - Tokenized inputs with generated unique example IDs.
#     """
#     questions = [q.strip() for q in examples["question"]]
#     inputs = tokenizer(
#         questions,
#         examples["context"],
#         max_length=max_length,
#         truncation="only_second",
#         stride=stride,
#         return_overflowing_tokens=True,
#         return_offsets_mapping=True,
#         padding="max_length",
#     )

#     sample_map = inputs.pop("overflow_to_sample_mapping")
#     #global example_ids

#     # Counter for unique ID generation
#     current_id = unique_id_start

#     for i in range(len(inputs["input_ids"])):
#         # Generate a unique ID
#         unique_id = f"unique-{current_id}"
#         current_id += 1
#         example_ids.append(unique_id)  # Append the generated unique ID

#         sequence_ids = inputs.sequence_ids(i)
#         offset = inputs["offset_mapping"][i]
#         inputs["offset_mapping"][i] = [
#             o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
#         ]

#     inputs["example_id"] = example_ids
#     return inputs

In [245]:
validation_dataset = raw_datasets["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)
len(raw_datasets["validation"]), len(validation_dataset)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

(16, 17)

In [246]:
validation_dataset 

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 17
})

In [247]:
small_eval_set = raw_datasets["validation"].select(range(4))
trained_checkpoint = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(trained_checkpoint)
eval_set = small_eval_set.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [248]:
eval_set

Dataset({
    features: ['input_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 4
})

In [249]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [250]:
import tensorflow as tf
from transformers import TFAutoModelForQuestionAnswering

eval_set_for_model = eval_set.remove_columns(["example_id", "offset_mapping"])
eval_set_for_model.set_format("numpy")

batch = {k: eval_set_for_model[k] for k in eval_set_for_model.column_names}
trained_model = TFAutoModelForQuestionAnswering.from_pretrained(trained_checkpoint)

outputs = trained_model(**batch)

All PyTorch model weights were used when initializing TFDistilBertForQuestionAnswering.

All the weights of TFDistilBertForQuestionAnswering were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForQuestionAnswering for predictions without further training.


In [251]:
start_logits = outputs.start_logits.numpy()
end_logits = outputs.end_logits.numpy()

In [252]:
import collections

example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(eval_set):
    example_to_features[feature["example_id"]].append(idx)

In [253]:
small_eval_set

Dataset({
    features: ['question', 'answers', 'context', 'answer_idx', 'id', '__index_level_0__'],
    num_rows: 4
})

In [254]:
import numpy as np

n_best = 20
max_answer_length = 30
predicted_answers = []

for example in small_eval_set:
    example_id = example["id"]
    context = example["context"]
    answers = []

    for feature_index in example_to_features[example_id]:
        start_logit = start_logits[feature_index]
        end_logit = end_logits[feature_index]
        offsets = eval_set["offset_mapping"][feature_index]

        start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
        end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
        for start_index in start_indexes:
            for end_index in end_indexes:
                # Skip answers that are not fully in the context
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue
                # Skip answers with a length that is either < 0 or > max_answer_length.
                if (
                    end_index < start_index
                    or end_index - start_index + 1 > max_answer_length
                ):
                    continue

                answers.append(
                    {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                )

    best_answer = max(answers, key=lambda x: x["logit_score"])
    predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"]})

In [255]:
import evaluate

metric = evaluate.load("squad")

RuntimeError: Failed to import transformers.pipelines because of the following error (look up to see its traceback):
This version of TensorFlow Probability requires TensorFlow version >= 2.14; Detected an installation of version 2.11.0. Please upgrade TensorFlow to proceed.

In [256]:
theoretical_answers = [
    {"id": ex["id"], "answers": ex["answers"]} for ex in small_eval_set
]

In [257]:
print(predicted_answers[0])
print(theoretical_answers[0])

{'id': 'unique-1', 'prediction_text': 'primary evaluative criterion in building design, construction, and operational strategies'}
{'id': 'unique-1', 'answers': 'Human health is prioritized to ensure that building design, construction, and operational strategies foster high-performance healing environments for patients, caregivers, and staff.'}


In [258]:
metric.compute(predictions=predicted_answers, references=theoretical_answers)

NameError: name 'metric' is not defined

In [185]:
# answers = raw_datasets.iloc[2:6]["answer_idx"].tolist()
# start_positions = []
# end_positions = []

# for i, offset in enumerate(inputs["offset_mapping"]):
#     sample_idx = inputs["overflow_to_sample_mapping"][i]
#     answer = answers[sample_idx]
#     start_char = answer["answer_start"][0]
#     end_char = answer["answer_start"][0] + len(answer["text"][0])
#     sequence_ids = inputs.sequence_ids(i)

#     # Find the start and end of the context
#     idx = 0
#     while sequence_ids[idx] != 1:
#         idx += 1
#     context_start = idx
#     while sequence_ids[idx] == 1:
#         idx += 1
#     context_end = idx - 1

#     # If the answer is not fully inside the context, label is (0, 0)
#     if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
#         start_positions.append(0)
#         end_positions.append(0)
#     else:
#         # Otherwise it's the start and end token positions
#         idx = context_start
#         while idx <= context_end and offset[idx][0] <= start_char:
#             idx += 1
#         start_positions.append(idx - 1)

#         idx = context_end
#         while idx >= context_start and offset[idx][1] >= end_char:
#             idx -= 1
#         end_positions.append(idx + 1)

# start_positions, end_positions

In [143]:
idx = 0
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

start = start_positions[idx]
end = end_positions[idx]
labeled_answer = tokenizer.decode(inputs["input_ids"][idx][start : end + 1])

print(f"Theoretical answer: {answer}, labels give: {labeled_answer}")

Theoretical answer: Building height is measured vertically from the average elevation of the finished grade to the topmost section of the roof., labels give: any structure with a roof supported by walls or columns, used for residence, business, industry, or other public or private


In [144]:
idx = 0
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

start = start_positions[idx]
end = end_positions[idx]
labeled_answer = tokenizer.decode(inputs["input_ids"][idx][start : end + 1])

print(f"Theoretical answer: {answer}, labels give: {labeled_answer}")

Theoretical answer: Building height is measured vertically from the average elevation of the finished grade to the topmost section of the roof., labels give: any structure with a roof supported by walls or columns, used for residence, business, industry, or other public or private


In [146]:
import pandas as pd

# Example settings
max_length = 384
stride = 128

# Function to process a single example
def preprocess_training_example(row, tokenizer):
    question = row["question"].strip()
    context = row["context"]
    answer = row["answer_idx"]
    inputs = tokenizer(
        question,
        context,
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    # Adding start and end positions back to inputs
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Example DataFrame (convert DatasetDict into a DataFrame)
raw_datasets_train = raw_datasets

# Process each row of the DataFrame
processed_data = raw_datasets_train.apply(
    lambda row: preprocess_training_example(row, tokenizer),
    axis=1
)

# Convert the results into a DataFrame or Series
train_dataset_df = pd.DataFrame(processed_data.tolist())

In [147]:
train_dataset_df

,attention_mask,end_positions,input_ids,start_positions,token_type_ids
0,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[28],"[[101, 1327, 1110, 170, 8315, 4267, 28042, 102...",[10],"[[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,..."
1,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[59],"[[101, 1327, 1110, 1103, 3719, 1206, 170, 107,...",[29],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
2,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[36],"[[101, 1327, 1110, 1103, 107, 1459, 3976, 107,...",[13],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1,..."
3,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[82],"[[101, 1327, 1674, 1103, 1858, 107, 1901, 107,...",[24],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
4,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[41],"[[101, 1327, 14684, 4912, 1538, 1129, 1316, 11...",[24],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
...,...,...,...,...,...
59,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[73],"[[101, 1327, 1547, 3333, 1191, 170, 139, 2137,...",[33],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
60,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[58],"[[101, 1327, 1674, 107, 11019, 10913, 1200, 20...",[18],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
61,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[33],"[[101, 1327, 4454, 1132, 3417, 1962, 1120, 166...",[27],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,..."
62,"[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",[69],"[[101, 1327, 1110, 1103, 1514, 3007, 1104, 110...",[27],"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


In [148]:
# from sklearn.model_selection import train_test_split

# # Split the data into training and validation sets
# train_df, validation_df = train_test_split(
#     train_dataset_df,  # The DataFrame containing the processed data
#     test_size=0.2,  # Percentage of the data to be used for validation (e.g., 20%)
#     random_state=42,  # For reproducibility
#     shuffle=True  # Shuffle the data before splitting
# )

# # Split the validation set into validation and test sets
# validation_df, test_df = train_test_split(
#     validation_df,  # The previously created validation DataFrame
#     test_size=0.2,  # Split 50% of the validation set into test data
#     random_state=42,  # For reproducibility
#     shuffle=True  # Shuffle the data before splitting
# )

# # Display the sizes of the splits
# print(f"Training set size: {len(train_df)}")
# print(f"Validation set size: {len(validation_df)}")
# print(f"Test set size: {len(test_df)}")

In [150]:
raw_datasets_val

,question,answers,context,answer_idx
28,How does the Sensitive Land Protection credit ...,By awarding points to projects that avoid envi...,The Sensitive Land Protection credit is anothe...,{'text': ['By awarding points to projects that...
68,What activities can take place in a community ...,Growing food and ornamental crops for personal...,Community gardens are areas used for growing f...,{'text': ['Growing food and ornamental crops f...
35,"What are access trails, and what features migh...",Access trails are pedestrian pathways designed...,"The term ""abut"" refers to properties or land f...",{'text': ['Access trails are pedestrian pathwa...
33,What additional steps are allowed for managing...,Hazardous or dying trees can be removed based ...,For projects on previously developed land or t...,{'text': ['Hazardous or dying trees can be rem...
4,What is the purpose of the preliminary LEED me...,What is the purpose of the preliminary LEED mT...,This integrative process seeks to incorporate ...,{'text': ['What is the purpose of the prelimin...
12,What is the ultimate goal of the integrative p...,The ultimate goal is to create a healthier bui...,"As part of this integrative process, a compreh...",{'text': ['The ultimate goal is to create a he...
22,In what ways might multifunctional spaces cont...,"By incorporating multifunctional spaces, proje...",Energy-related systems are a core focus within...,{'text': ['By incorporating multifunctional sp...
45,How does articulation enhance the visual appea...,Articulation adds variety and visual interest ...,An Area of Special Flood Hazard is land in the...,{'text': ['Articulation adds variety and visua...
18,What role does the water budget analysis play ...,The water budget analysis is crucial for ident...,"In addition to energy, water-related systems r...",{'text': ['The water budget analysis is crucia...
70,What is the purpose of a connection in stormwa...,"To divert or transmit stormwater drainage, imp...","A connection is a structure, such as a ditch o...",{'text': ['To divert or transmit stormwater dr...


In [156]:
import uuid

def preprocess_validation_examples_df(df, tokenizer, max_length, stride):
    """
    Preprocess the validation examples for DataFrame input.

    Parameters:
    - df: pd.DataFrame containing 'id', 'question', and 'context'.
    - tokenizer: Tokenizer object to process text.
    - max_length: Maximum token sequence length.
    - stride: Stride length for overlapping tokens.

    Returns:
    - A dictionary with tokenized inputs including offset_mapping and unique example_ids.
    """
    questions = [q.strip() for q in df["question"]]
    contexts = df["context"].tolist()

    # Tokenize the inputs
    inputs = tokenizer(
        questions,
        contexts,
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    # Counter for generating unique IDs
    unique_id_counter = 1

    # Process tokenized inputs
    for i in range(len(inputs["input_ids"])):
        # Generate a unique ID
        unique_id = f"example-{unique_id_counter}"
        unique_id_counter += 1
        example_ids.append(unique_id)

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs


# Example usage:
validation_df = raw_datasets_val  # Convert validation set to DataFrame

# Apply preprocessing
processed_validation = preprocess_validation_examples_df(
    validation_df,
    tokenizer=tokenizer,
    max_length=max_length,
    stride=stride,
)

# Convert the processed outputs into a DataFrame for further use
processed_validation_df = pd.DataFrame(processed_validation)

In [161]:
processed_validation

{'input_ids': [[101, 1731, 1674, 1103, 14895, 22472, 4026, 8063, 4755, 1494, 4851, 14769, 3772, 136, 102, 1109, 14895, 22472, 4026, 8063, 4755, 1110, 1330, 1696, 6552, 1104, 1103, 149, 1942, 6459, 117, 25055, 122, 1106, 123, 1827, 1106, 3203, 1115, 3644, 4801, 1193, 7246, 4508, 117, 2456, 8715, 25596, 1103, 14769, 3772, 1104, 1718, 119, 1109, 4755, 3272, 1160, 6665, 131, 25338, 14829, 1113, 2331, 1872, 1657, 1137, 11027, 1657, 1115, 1674, 1136, 2283, 9173, 1111, 15750, 117, 1216, 1112, 5748, 17790, 117, 7870, 18220, 1116, 117, 1137, 10902, 1111, 11532, 1530, 119, 1188, 3636, 8559, 1106, 1447, 3470, 1105, 20432, 117, 8753, 170, 5867, 20232, 1104, 1620, 1623, 1111, 1447, 3470, 1105, 1851, 1623, 1111, 20432, 119, 21454, 14485, 1111, 1142, 4755, 1538, 1129, 5784, 1106, 3644, 1657, 3626, 1112, 3607, 1111, 6487, 117, 23600, 117, 1137, 1447, 23406, 119, 14895, 22472, 10902, 1132, 3393, 1118, 7225, 1176, 1103, 158, 119, 156, 119, 5135, 27290, 1174, 11763, 2173, 117, 7009, 1708, 22552, 5393, 11

In [ ]:
# def preprocess_validation_examples(examples):
#     questions = [q.strip() for q in examples["question"]]
#     inputs = tokenizer(
#         questions,
#         examples["context"],
#         max_length=max_length,
#         truncation="only_second",
#         stride=stride,
#         return_overflowing_tokens=True,
#         return_offsets_mapping=True,
#         padding="max_length",
#     )

#     sample_map = inputs.pop("overflow_to_sample_mapping")
#     example_ids = []

#     for i in range(len(inputs["input_ids"])):
#         sample_idx = sample_map[i]
#         example_ids.append(examples["id"][sample_idx])

#         sequence_ids = inputs.sequence_ids(i)
#         offset = inputs["offset_mapping"][i]
#         inputs["offset_mapping"][i] = [
#             o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
#         ]

#     inputs["example_id"] = example_ids
#     return inputs

# validation_dataset = raw_datasets["validation"].map(
#     preprocess_validation_examples,
#     batched=True,
#     remove_columns=raw_datasets["validation"].column_names,
# )
# len(raw_datasets["validation"]), len(validation_dataset)

In [152]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/home/rkannan/miniconda3/envs/richard_tf/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [102]:
validation_df.shape

(12, 5)

In [104]:
small_eval_set = validation_df[0:3]
trained_checkpoint = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(trained_checkpoint)

In [105]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [128]:
import tensorflow as tf
from transformers import TFAutoModelForQuestionAnswering

eval_set = small_eval_set
eval_set_for_model = eval_set.drop(['end_positions','start_positions','token_type_ids'],axis=1)
eval_set_for_model.to_numpy()

batch = {k: np.array(eval_set_for_model[k].tolist())[0] for k in eval_set_for_model.columns}
trained_model = TFAutoModelForQuestionAnswering.from_pretrained(trained_checkpoint)

outputs = trained_model(**batch)

/home/rkannan/miniconda3/envs/richard_tf/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
All PyTorch model weights were used when initializing TFDistilBertForQuestionAnswering.

All the weights of TFDistilBertForQuestionAnswering were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForQuestionAnswering for predictions without further training.


In [129]:
start_logits = outputs.start_logits.numpy()
end_logits = outputs.end_logits.numpy()

In [130]:
import collections

example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(eval_set):
    example_to_features[feature["example_id"]].append(idx)

TypeError: string indices must be integers